### Import the CKAN and Parsing Tools
Load the libraries used to query CKAN, fetch measurement resources, and write the station CSV files used in later notebooks.


In [1]:
import json
from pathlib import Path

import pandas as pd
import requests


# USGS Compaction Data Ingest
This notebook now uses the CKAN datasets published from Upstream instead of scraping ScienceBase directly. It queries CKAN for the Houston-area extensometer campaign station datasets, reads each measurement resource, and writes the same `csv_files/` outputs expected by the later Folium notebooks.

Campaign search tag: `houston-area-extensometer-compaction-campaign`
CKAN catalog: https://ckan.tacc.utexas.edu/


### Query CKAN for the Station Datasets
Use the CKAN `package_search` API to find all station datasets published for this campaign. Each dataset contains a GeoJSON measurement resource for one extensometer station.


In [2]:
CKAN_SEARCH_URL = 'https://ckan.tacc.utexas.edu/api/3/action/package_search'
CAMPAIGN_TAG = 'houston-area-extensometer-compaction-campaign'
params = {
    'q': f'tags:{CAMPAIGN_TAG} AND tags:upstream',
    'rows': 100,
}
response = requests.get(CKAN_SEARCH_URL, params=params, timeout=30)
response.raise_for_status()
payload = response.json()
station_packages = payload['result']['results']
station_packages = sorted(
    station_packages,
    key=lambda pkg: int(next(extra['value'] for extra in pkg.get('extras', []) if extra['key'] == 'station_id'))
)
len(station_packages)


14

### Inspect the CKAN Results
Preview the station datasets and the measurement resource URL that will be fetched for each station.


In [3]:
preview_rows = []
for pkg in station_packages:
    measurement_resource = next(
        resource for resource in pkg['resources']
        if resource.get('format') == 'GeoJSON' and resource['name'].endswith('-measurements')
    )
    station_id = next(extra['value'] for extra in pkg.get('extras', []) if extra['key'] == 'station_id')
    preview_rows.append({
        'station_id': int(station_id),
        'title': pkg['title'],
        'measurement_url': measurement_resource['url'],
    })
pd.DataFrame(preview_rows)


,station_id,title,measurement_url
0,4,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...
1,5,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...
2,6,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...
3,7,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...
4,8,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...
5,9,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...
6,10,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...
7,11,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...
8,12,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...
9,13,Houston-area extensometer compaction campaign ...,https://upstreamapi.pods.portals.tapis.io/api/...


### Create the Output Folder
Create the `csv_files/` directory if it does not already exist. The notebook writes the same station CSV outputs used by the follow-on notebooks.


In [4]:
csv_dir = Path('csv_files')
csv_dir.mkdir(exist_ok=True)
csv_dir


PosixPath('csv_files')

### Fetch the CKAN Measurement Resources and Rebuild the CSV Files
For each station dataset, fetch the measurement resource from Upstream, convert it to the legacy two-column CSV format, and rebuild `TABLE1_CompactionSites.csv` from the CKAN metadata.


In [5]:
TABLE_EXPORTS = {
    '292458094534206': {'table': 'TABLE3_TexasCity_2023.csv', 'general_name': 'Texas City'},
    '295449095084105': {'table': 'TABLE8_LakeHouston_2023.csv', 'general_name': 'Lake Houston'},
    '294726095351102': {'table': 'TABLE2_Addicks_2023.csv', 'general_name': 'Addicks'},
    '294728095200106': {'table': 'TABLE7_Northeast_2023.csv', 'general_name': 'Northeast'},
    '294527095014910': {'table': 'TABLE13_BaytownC1_Shallow_2023.csv', 'general_name': 'Baytown Shallow'},
    '294527095014911': {'table': 'TABLE14_BaytownC2_Deep_2023.csv', 'general_name': 'Baytown Deep'},
    '294338095270402': {'table': 'TABLE4_Southwest_2023.csv', 'general_name': 'Southwest'},
    '294206095162601': {'table': 'TABLE10_EastEnd_2023.csv', 'general_name': 'East End'},
    '294237095093204': {'table': 'TABLE6_Pasadena_2023.csv', 'general_name': 'Pasadena'},
    '293306095054101': {'table': 'TABLE9_JohnsonSpaceCenter_NASA_2023.csv', 'general_name': 'Johnson Space Center'},
    '293349095070901': {'table': 'TABLE11_ClearLakeShallow_2023.csv', 'general_name': 'Clear Lake Shallow'},
    '293348095070604': {'table': 'TABLE12_ClearLakeDeep_2023.csv', 'general_name': 'Clear Lake Deep'},
    '293352095011601': {'table': 'TABLE5_Seabrook_2023.csv', 'general_name': 'Seabrook'},
    '294327095445201': {'table': 'TABLE15_FortBend_CincoMUD_2023.csv', 'general_name': 'Fort Bend'},
}

site_rows = []
written_files = []

def extract_lon_lat(spatial):
    coordinates = spatial['coordinates']
    if spatial.get('type') == 'Point':
        return coordinates

    # Some station footprints are stored as tiny polygons around the site.
    # Use the exterior ring centroid so downstream point mapping still works.
    if spatial.get('type') == 'Polygon':
        ring = coordinates[0]
        ring_points = ring[:-1] or ring
        longitudes = [point[0] for point in ring_points]
        latitudes = [point[1] for point in ring_points]
        return sum(longitudes) / len(longitudes), sum(latitudes) / len(latitudes)

    raise ValueError(f"Unsupported spatial geometry type: {spatial.get('type')}")

for pkg in station_packages:
    extras = {extra['key']: extra['value'] for extra in pkg.get('extras', [])}
    station_name = extras['station_name']
    site_no = pkg['notes'].split('source_site_no=')[1].strip()
    export_info = TABLE_EXPORTS[site_no]
    spatial = json.loads(pkg['spatial'])
    longitude, latitude = extract_lon_lat(spatial)
    interval = pkg['notes'].split('interval=')[1].split(';')[0].strip()
    anchor_depth = pkg['notes'].split('anchor_depth=')[1].split(' ft')[0].strip()

    site_rows.append({
        'SITE_NO': site_no,
        'STATION_NM': station_name,
        'COMPACTION_INTERVAL': interval,
        'ANCHOR_DEPTH': anchor_depth,
        'DEC_LONG_VA': longitude,
        'DEC_LAT_VA': latitude,
        'GENERAL_NM': export_info['general_name'],
    })

    measurement_resource = next(
        resource for resource in pkg['resources']
        if resource.get('format') == 'GeoJSON' and resource['name'].endswith('-measurements')
    )
    measurement_response = requests.get(measurement_resource['url'], timeout=60)
    measurement_response.raise_for_status()
    measurement_items = measurement_response.json()['items']

    df = pd.DataFrame(measurement_items)[['collectiontime', 'value']].rename(
        columns={'collectiontime': 'DATE', 'value': 'CUMULATIVE_COMPACTION'}
    )
    df['DATE'] = pd.to_datetime(df['DATE']).map(lambda dt: f'{dt.month}/{dt.day}/{dt.year}')
    df = df.sort_values('DATE', key=lambda s: pd.to_datetime(s)).reset_index(drop=True)

    out_path = csv_dir / export_info['table']
    df.to_csv(out_path, index=False)
    written_files.append(out_path.name)

sites_df = pd.DataFrame(site_rows)
sites_df['table_order'] = sites_df['SITE_NO'].map(lambda site_no: int(TABLE_EXPORTS[site_no]['table'].split('_')[0].replace('TABLE', '')))
sites_df = sites_df.sort_values('table_order').drop(columns='table_order')
sites_df.to_csv(csv_dir / 'TABLE1_CompactionSites.csv', index=False)
written_files.insert(0, 'TABLE1_CompactionSites.csv')
written_files


['TABLE1_CompactionSites.csv',
 'TABLE3_TexasCity_2023.csv',
 'TABLE8_LakeHouston_2023.csv',
 'TABLE2_Addicks_2023.csv',
 'TABLE7_Northeast_2023.csv',
 'TABLE13_BaytownC1_Shallow_2023.csv',
 'TABLE14_BaytownC2_Deep_2023.csv',
 'TABLE4_Southwest_2023.csv',
 'TABLE10_EastEnd_2023.csv',
 'TABLE6_Pasadena_2023.csv',
 'TABLE9_JohnsonSpaceCenter_NASA_2023.csv',
 'TABLE11_ClearLakeShallow_2023.csv',
 'TABLE12_ClearLakeDeep_2023.csv',
 'TABLE5_Seabrook_2023.csv',
 'TABLE15_FortBend_CincoMUD_2023.csv']

### Confirm the Generated Files
List the files written to `csv_files/` so you can verify that the expected CKAN-backed station tables were created.


In [6]:
sorted(path.name for path in csv_dir.glob('*.csv'))


['TABLE10_EastEnd_2023.csv',
 'TABLE11_ClearLakeShallow_2023.csv',
 'TABLE12_ClearLakeDeep_2023.csv',
 'TABLE13_BaytownC1_Shallow_2023.csv',
 'TABLE14_BaytownC2_Deep_2023.csv',
 'TABLE15_FortBend_CincoMUD_2023.csv',
 'TABLE1_CompactionSites.csv',
 'TABLE2_Addicks_2023.csv',
 'TABLE3_TexasCity_2023.csv',
 'TABLE4_Southwest_2023.csv',
 'TABLE5_Seabrook_2023.csv',
 'TABLE6_Pasadena_2023.csv',
 'TABLE7_Northeast_2023.csv',
 'TABLE8_LakeHouston_2023.csv',
 'TABLE9_JohnsonSpaceCenter_NASA_2023.csv']

### Preview the Station Metadata Table
Inspect the rebuilt `TABLE1_CompactionSites.csv` table before moving on to the notebook that combines all station measurements.


In [7]:
pd.read_csv(csv_dir / 'TABLE1_CompactionSites.csv')


,SITE_NO,STATION_NM,COMPACTION_INTERVAL,ANCHOR_DEPTH,DEC_LONG_VA,DEC_LAT_VA,GENERAL_NM
0,294726095351102,LJ-65-12-726 (Addicks Extensometer),CHICOT AND EVANGELINE,1802,-95.5861,29.79070,Addicks
1,292458094534206,KH-64-33-920 (Texas City Extensometer),CHICOT,800,-94.8950,29.41630,Texas City
2,294338095270402,LJ-65-21-226 (Southwest Extensometer),CHICOT AND EVANGELINE,2358,-95.4508,29.72700,Southwest
3,293352095011601,LJ-65-32-625 (Seabrook Extensometer),CHICOT,1381,-95.0215,29.56480,Seabrook
4,294237095093204,LJ-65-23-322 (Pasadena Extensometer),CHICOT AND EVANGELINE,2831,-95.1593,29.71020,Pasadena
5,294728095200106,LJ-65-14-746 (Northeast Extensometer),CHICOT AND EVANGELINE,2170,-95.3340,29.79090,Northeast
6,295449095084105,LJ-65-07-909 (Lake Houston Extensometer),CHICOT AND EVANGELINE,2592,-95.1455,29.91320,Lake Houston
7,293306095054101,LJ-65-32-401 (NASA Extensometer),CHICOT,770,-95.0960,29.55190,Johnson Space Center
8,294206095162601,LJ-65-22-622 (East End Extensometer),CHICOT,995,-95.2741,29.70170,East End
9,293349095070901,LJ-65-32-424 (Clear Lake Shallow Extensometer),CHICOT,1740,-95.1193,29.56380,Clear Lake Shallow


### Result
At this point, the station metadata table and the per-station compaction CSV files have been rebuilt from CKAN and Upstream resources in `csv_files/`. The next notebook can use them without any dependence on the original ScienceBase download step.
